In [1]:
import numpy as np
import pandas as pd


In [2]:
data = pd.read_csv('offers.csv')

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import linear_kernel 
documents = [
    "I love Project Hail Mary",
    "Project Hail Mary is amazing",
    "I enjoy Sci- fi Movies"
]

# #Step 1: Convert text to IF-IDF vectors 
# vectorizer = TfidfVectorizer()
# tfidf_matrix = vectorizer.fit_transform(data)

tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 3), min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(data['skills'])


In [4]:
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix) 
print(cosine_similarities)

[[1.         0.00520127 0.00692323 ... 0.00389422 0.00482137 0.00413152]
 [0.00520127 1.         0.04689    ... 0.0608416  0.07532698 0.14191227]
 [0.00692323 0.04689    1.         ... 0.03510685 0.04346521 0.00795898]
 ...
 [0.00389422 0.0608416  0.03510685 ... 1.         0.09071887 0.08557506]
 [0.00482137 0.07532698 0.04346521 ... 0.09071887 1.         0.0758511 ]
 [0.00413152 0.14191227 0.00795898 ... 0.08557506 0.0758511  1.        ]]


In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-distilroberta-base-v1')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
skills = data['skills'].tolist()
# print(descriptions)
skills_embeddings = []
for i,skls in enumerate(skills):
    skills_embeddings.append(model.encode(skls))

In [9]:
# skills_embeddings = np.load('skills_embeddings.npy')

EOFError: No data left in file

In [12]:
import torch
from sentence_transformers import SentenceTransformer, util

def recommend(query):
    #Compute cosine-similarities with all embeddings 
    query_embedd = model.encode(query)
    cosine_scores = util.pytorch_cos_sim(query_embedd, skills_embeddings)
    top5_matches = torch.argsort(cosine_scores, dim=-1, descending=True).tolist()[0][1:6]
    return top5_matches

In [18]:
id = 90
query_show_des = data.loc[data['user_id'] == id]['skills'].to_list()[0]
recommendded_results = recommend(query_show_des)

for index in recommendded_results:
    print(data.iloc[index,:])

user_id                                                          113
status                                                     published
city                                                          DORADO
job_title                               Corporate Reservations Agent
organization_id                                                  113
contracts                                                         AL
description        'Lorem ipsum dolor sit amet consectetur adipis...
skills             {'skills': {'hardware': 2, 'communication': 1,...
Name: 113, dtype: object
user_id                                                           42
status                                                     published
city                                                        SANDUSKY
job_title                                      Aviation Inside Sales
organization_id                                                   15
contracts                                                         AL
descripti